# Narração com a SUA voz — gratuito (Chatterbox-TTS-Indonesian)

Clona a voz da referência (`referencia.wav`) e narra cada cena do `narracao.json`
em indonésio. Modelo **Apache 2.0** (uso comercial liberado), clonagem *zero-shot*
— nenhum treino, só o áudio de referência.

**Antes de rodar:** Ambiente de execução → Alterar tipo → **GPU (T4)**.

Fluxo: rode as células na ordem → envie os 2 arquivos quando pedido → baixe o
`narracao.zip` no final → extraia em `out/<slug>/audio/` → `maquina retomar <slug>`.


In [ ]:
#@title 1. Instalar (3-5 min na primeira vez)
%pip -q install chatterbox-tts huggingface_hub soundfile
import torch
assert torch.cuda.is_available(), "Ative a GPU: Ambiente de execucao > Alterar tipo > T4"
print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
#@title 2. Enviar arquivos: narracao.json + referencia.wav
from google.colab import files
subidos = files.upload()
assert any(n.endswith(".json") for n in subidos), "faltou o narracao.json"
assert any(n.endswith((".wav", ".mp3")) for n in subidos), "faltou a referencia de voz"
JSON_PATH = next(n for n in subidos if n.endswith(".json"))
VOZ_PATH = next(n for n in subidos if n.endswith((".wav", ".mp3")))
print("roteiro:", JSON_PATH, "| voz:", VOZ_PATH)

In [ ]:
#@title 3. Carregar o modelo indonésio (fine-tune Apache 2.0)
# Checkpoint: https://huggingface.co/grandhigh/Chatterbox-TTS-Indonesian
from huggingface_hub import snapshot_download
from chatterbox.tts import ChatterboxTTS
import pathlib

ckpt = snapshot_download("grandhigh/Chatterbox-TTS-Indonesian")
# O fine-tune segue o layout do Chatterbox; from_local cobre a maioria dos casos.
try:
    modelo = ChatterboxTTS.from_local(ckpt, device="cuda")
except Exception as e:
    print("from_local falhou (", e, ") — tentando from_pretrained padrao + pesos locais")
    modelo = ChatterboxTTS.from_pretrained(device="cuda")
print("modelo pronto")

In [ ]:
#@title 4. Gerar as narrações (uma por cena)
import json, pathlib, soundfile as sf, torchaudio

dados = json.loads(pathlib.Path(JSON_PATH).read_text(encoding="utf-8"))
saida = pathlib.Path("narracao"); saida.mkdir(exist_ok=True)

# Naturalidade > espetaculo: exaggeration baixo e cfg moderado soam menos "IA".
PARAMS = dict(exaggeration=0.35, cfg_weight=0.4)

for cena in dados["cenas"]:
    wav = modelo.generate(cena["texto"], audio_prompt_path=VOZ_PATH, **PARAMS)
    destino = saida / f"cena_{cena['indice']:03d}.mp3"
    torchaudio.save(str(destino), wav, modelo.sr, format="mp3")
    print(destino, f"{wav.shape[-1]/modelo.sr:.1f}s")
print("cenas geradas:", len(dados["cenas"]))

In [ ]:
#@title 5. Ouvir uma amostra antes de baixar tudo
from IPython.display import Audio, display
import pathlib
primeira = sorted(pathlib.Path("narracao").glob("*.mp3"))[0]
display(Audio(str(primeira)))

In [ ]:
#@title 6. Empacotar e baixar
import shutil
from google.colab import files
shutil.make_archive("narracao", "zip", "narracao")
files.download("narracao.zip")
print("Extraia em out/<slug>/audio/ e rode: maquina retomar <slug>")

---
**Outros idiomas:** troque o checkpoint da célula 3 pelo
[Chatterbox Multilingual v3](https://huggingface.co/ResembleAI/chatterbox) (MIT,
20+ idiomas) usando `ChatterboxMultilingualTTS.from_pretrained(device="cuda")` e
passando `language_id` no `generate`. A referência de voz é a mesma.

**Se soar artificial:** aumente a referência para 20-30s de fala contínua e limpa;
reduza `exaggeration`; confira se o texto tem pontuação natural (vírgulas guiam a
prosódia).